# NutriLens AI: Notebook 4 — Comprehensive Model Evaluation & Artifact Export

**Project:** NutriLens AI — AI-Powered Food Recognition and Nutrition Analysis  
**Author:** NutriLens AI Engineering Team  
**Dataset:** `SohlHealth/enhanced-indian-food-classification` (Unseen Test Split: 2,311 images, 43 classes)  
**Target Environment:** Google Colab (with GPU acceleration) / Local Jupyter

---

### Objectives:
1. **Load Checkpoint & Unseen Test Split:** Load the trained model checkpoint (`best_model.pt`) and evaluate against the holdout test set (15% split, 2,311 samples).
2. **Comprehensive Metrics:**
   - Top-1 and Top-5 Classification Accuracy.
   - Precision, Recall, and F1-Score (Macro and Weighted averages).
3. **Confusion Matrix Analysis:** Plot a high-resolution 43x43 normalized confusion matrix heatmap to identify classification ambiguities between visually similar dishes (e.g., `dal_makhani` vs. `rajma_curry`).
4. **Per-Class Performance Breakdown:** Tabulate and visualize precision, recall, and F1-scores across all 43 Indian food categories.
5. **Qualitative Evaluation on Unseen Images:** Visualize real test images with ground truth, predicted class, confidence scores, and Top-5 candidate probability distributions.
6. **Artifact Export & Verification:** Package model weights, class names, preprocessing configuration, and version metadata into a standardized distribution bundle for the Streamlit application.


## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install --quiet datasets huggingface_hub pyarrow pillow matplotlib seaborn pandas scikit-learn tqdm torch torchvision tabulate

import os
import sys
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Load Trained Model Checkpoint & Metadata

In [ ]:
CHECKPOINT_PATH = "../models/best_model.pt"
CLASS_NAMES_PATH = "../models/class_names.json"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint not found at: {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
arch = checkpoint.get("architecture", "mobilenet_v2")
classes = checkpoint.get("classes", [])
num_classes = len(classes)

print(f"Loaded Checkpoint Metadata:")
print(f"  - Architecture: {arch}")
print(f"  - Number of Classes: {num_classes}")
print(f"  - Input Dimensions: {checkpoint.get('input_size', [3, 224, 224])}")
print(f"  - Validation Loss in Checkpoint: {checkpoint.get('val_loss', 'N/A')}")
print(f"  - Validation Top-1 Acc in Checkpoint: {checkpoint.get('val_top1', 'N/A')}")

# Rebuild Model Architecture
if arch == "mobilenet_v2":
    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
elif arch == "resnet50":
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
else:
    raise ValueError(f"Unknown architecture: {arch}")

# Load state dict
if "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()
print(f"\nModel successfully instantiated on {device} in evaluation mode.")


## 3. Prepare Holdout Test Dataset (2,311 Unseen Images)

In [ ]:
from datasets import load_dataset, concatenate_datasets

DATASET_REPO = "SohlHealth/enhanced-indian-food-classification"
print("Loading dataset from Hugging Face for test evaluation...")
raw_ds = load_dataset(DATASET_REPO)
pooled_dataset = concatenate_datasets([raw_ds[s] for s in raw_ds.keys()])

all_labels = [item["label"] for item in pooled_dataset]
all_indices = np.arange(len(pooled_dataset))
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for i, c in enumerate(classes)}

# Re-create exact holdout test split (random_state=42)
SEED = 42
_, temp_idx, _, temp_y = train_test_split(
    all_indices, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED
)
_, test_idx, _, test_y = train_test_split(
    temp_idx, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED
)

print(f"Holdout Test Set Size: {len(test_idx):,} images across {len(set(test_y))} classes")

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class IndianFoodDataset(Dataset):
    def __init__(self, dataset, indices, class_to_idx, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        sample = self.dataset[real_idx]
        image = sample["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")
        label = self.class_to_idx[sample["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = IndianFoodDataset(pooled_dataset, test_idx, class_to_idx, test_transform)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2 if device.type == "cuda" else 0,
    pin_memory=(device.type == "cuda")
)


## 4. Run Inference on Unseen Test Set & Compute Global Metrics

In [ ]:
all_preds = []
all_targets = []
all_probs = []
correct_top1 = 0
correct_top5 = 0
total_samples = 0

criterion = nn.CrossEntropyLoss()
total_test_loss = 0.0

print("Evaluating model across all test samples...")
with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Running Test Evaluation"):
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        loss = criterion(outputs, targets)
        total_test_loss += loss.item() * images.size(0)

        probs = torch.softmax(outputs, dim=1)
        
        # Top-1
        _, preds = torch.max(outputs, 1)
        correct_top1 += (preds == targets).sum().item()

        # Top-5
        _, top5_preds = outputs.topk(min(5, outputs.size(1)), dim=1, largest=True, sorted=True)
        correct_top5 += top5_preds.eq(targets.view(-1, 1).expand_as(top5_preds)).sum().item()

        total_samples += targets.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

avg_test_loss = total_test_loss / total_samples
top1_acc = correct_top1 / total_samples
top5_acc = correct_top5 / total_samples

# Precision, Recall, F1
prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)
prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(all_targets, all_preds, average="weighted", zero_division=0)

print("\n" + "="*60)
print("TEST SET EVALUATION RESULTS (2,311 Unseen Images):")
print("="*60)
print(f"  - Test Loss:             {avg_test_loss:.4f}")
print(f"  - Top-1 Accuracy:        {top1_acc*100:.2f}%")
print(f"  - Top-5 Accuracy:        {top5_acc*100:.2f}%")
print(f"  - Macro Precision:       {prec_macro*100:.2f}%")
print(f"  - Macro Recall:          {rec_macro*100:.2f}%")
print(f"  - Macro F1-Score:        {f1_macro*100:.2f}%")
print(f"  - Weighted Precision:    {prec_weighted*100:.2f}%")
print(f"  - Weighted Recall:       {rec_weighted*100:.2f}%")
print(f"  - Weighted F1-Score:     {f1_weighted*100:.2f}%")
print("="*60)


## 5. Normalized Confusion Matrix Heatmap (43 Classes)

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm_norm,
    annot=False,
    fmt=".2f",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.title(f"NutriLens AI — Normalized Confusion Matrix (43 Classes, Top-1 Acc: {top1_acc*100:.1f}%)", fontsize=15, fontweight="bold")
plt.xlabel("Predicted Class", fontsize=12)
plt.ylabel("True Class", fontsize=12)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# Detect most confused class pairs (excluding diagonal)
np.fill_diagonal(cm_norm, 0)
confused_pairs = []
for i in range(num_classes):
    for j in range(num_classes):
        if cm_norm[i, j] > 0.05:  # more than 5% misclassification rate
            confused_pairs.append({
                "True Class": classes[i],
                "Confused As": classes[j],
                "Error Rate": cm_norm[i, j]
            })

df_confused = pd.DataFrame(confused_pairs).sort_values(by="Error Rate", ascending=False)
print("\nTop Most Confused Indian Food Class Pairs:")
print(df_confused.head(10).to_string(index=False))


## 6. Per-Class Precision, Recall, and F1-Score Analysis

In [ ]:
report = classification_report(all_targets, all_preds, target_names=classes, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).transpose().reset_index()
df_classes = df_report[df_report["index"].isin(classes)].rename(columns={"index": "Food Class"}).sort_values(by="f1-score", ascending=False)

print("Top 5 Best-Performing Indian Food Classes:")
print(df_classes.head(5)[["Food Class", "precision", "recall", "f1-score", "support"]].to_string(index=False))

print("\nTop 5 Most Challenging Indian Food Classes:")
print(df_classes.tail(5)[["Food Class", "precision", "recall", "f1-score", "support"]].to_string(index=False))

# Horizontal bar chart of per-class F1-scores
plt.figure(figsize=(12, 14))
sns.barplot(data=df_classes, x="f1-score", y="Food Class", palette="mako")
plt.title("Per-Class F1-Score Performance (43 Classes)", fontsize=14, fontweight="bold")
plt.xlabel("F1-Score", fontsize=12)
plt.ylabel("Food Class", fontsize=12)
plt.axvline(f1_macro, color="red", linestyle="--", label=f"Macro Avg F1: {f1_macro:.2f}")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 7. Qualitative Predictions on Unseen Test Images with Confidence Bars

In [ ]:
# Select 6 diverse test images for qualitative inspection
sample_test_indices = [0, 50, 100, 250, 500, 750]

fig, axes = plt.subplots(len(sample_test_indices), 2, figsize=(14, 3 * len(sample_test_indices)))

for i, test_i in enumerate(sample_test_indices):
    real_sample_idx = int(test_idx[test_i])
    sample_item = pooled_dataset[real_sample_idx]
    orig_img = sample_item["image"]
    true_label = sample_item["label"]

    probs = all_probs[test_i]
    top5_idx = np.argsort(probs)[-5:][::-1]
    top5_classes = [classes[idx] for idx in top5_idx]
    top5_vals = [probs[idx] for idx in top5_idx]

    pred_label = top5_classes[0]
    is_correct = (pred_label == true_label)

    # Image subplot
    axes[i, 0].imshow(orig_img)
    color = "green" if is_correct else "red"
    axes[i, 0].set_title(
        f"True: {true_label}\nPred: {pred_label} ({top5_vals[0]*100:.1f}%)",
        color=color,
        fontsize=11,
        fontweight="bold"
    )
    axes[i, 0].axis("off")

    # Bar chart subplot
    y_pos = np.arange(len(top5_classes))
    bar_colors = ["green" if c == true_label else "steelblue" for c in top5_classes]
    axes[i, 1].barh(y_pos, top5_vals, color=bar_colors)
    axes[i, 1].set_yticks(y_pos)
    axes[i, 1].set_yticklabels(top5_classes, fontsize=10)
    axes[i, 1].invert_yaxis()
    axes[i, 1].set_xlim(0, 1)
    axes[i, 1].set_xlabel("Predicted Probability")
    axes[i, 1].grid(axis="x", linestyle="--", alpha=0.5)

plt.suptitle("Qualitative Test Set Predictions (Green = True Class)", fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()


## 8. Export Metrics, Preprocessing Config & Version Manifest

In [ ]:
os.makedirs("../data", exist_ok=True)
os.makedirs("../models", exist_ok=True)

# 1. Export evaluation_metrics.json
metrics_payload = {
    "model_architecture": arch,
    "test_set_size": total_samples,
    "num_classes": num_classes,
    "overall_metrics": {
        "test_loss": round(float(avg_test_loss), 4),
        "top1_accuracy": round(float(top1_acc), 4),
        "top5_accuracy": round(float(top5_acc), 4),
        "macro_precision": round(float(prec_macro), 4),
        "macro_recall": round(float(rec_macro), 4),
        "macro_f1": round(float(f1_macro), 4),
        "weighted_precision": round(float(prec_weighted), 4),
        "weighted_recall": round(float(rec_weighted), 4),
        "weighted_f1": round(float(f1_weighted), 4)
    },
    "top_confused_pairs": df_confused.head(10).to_dict(orient="records"),
    "per_class_f1": df_classes.set_index("Food Class")["f1-score"].round(4).to_dict()
}

with open("../data/evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print("Exported ../data/evaluation_metrics.json")

# 2. Export models/version.json
version_payload = {
    "project": "NutriLens AI",
    "version": "1.0.0",
    "exported_at": "2026-11-08",
    "framework": f"PyTorch {torch.__version__}",
    "model_architecture": arch,
    "weights_file": "best_model.pt",
    "class_names_file": "class_names.json",
    "model_config_file": "model_config.json",
    "metrics_file": "evaluation_metrics.json",
    "input_resolution": [224, 224],
    "top1_accuracy": round(float(top1_acc), 4),
    "top5_accuracy": round(float(top5_acc), 4)
}

with open("../models/version.json", "w", encoding="utf-8") as f:
    json.dump(version_payload, f, indent=2)

print("Exported ../models/version.json")
print("\nNotebook 4 Evaluation & Export Complete! All artifacts validated for Streamlit deployment.")
